# Store Sales Competetion


# Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
import pandas as pd
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
# Events
# Step 1: Load all required CSV files
train = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/train.csv')
stores = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/stores.csv')
oil = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/oil.csv')
holidays = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/holidays_events.csv')
transactions = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/transactions.csv')

# Step 2: Merge stores.csv (based on 'store_nbr')
train = train.merge(stores, on='store_nbr', how='left')

# Step 3: Merge oil.csv (based on 'date')
train = train.merge(oil[['date', 'dcoilwtico']], on='date', how='left')

# Step 4: Prepare and merge holidays_events.csv
# Create a flag column
holidays['holiday_flag'] = 1

# Optionally filter for only real holidays
# holidays = holidays[(holidays['type'] == 'Holiday') & (holidays['locale'] == 'National')]

# Merge with train
train = train.merge(holidays[['date', 'holiday_flag']], on='date', how='left')

# Step 5: Merge transactions.csv
train = train.merge(transactions, on=['date', 'store_nbr'], how='left')

# Step 6: Fill missing values
# 6.1 Fill holiday_flag with 0 (non-holiday)
train['holiday_flag'] = train['holiday_flag'].fillna(0)

# 6.2 Fill transactions with 0 (assume no transaction info = no transactions)
train['transactions'] = train['transactions'].fillna(0)

# 6.3 Use Iterative Imputer for dcoilwtico (oil price)
def fill_oil_with_iterative_imputer(df):
    imputer = IterativeImputer(random_state=42)
    cols_for_imputation = ['dcoilwtico', 'transactions', 'sales', 'onpromotion']
    used_cols = [col for col in cols_for_imputation if col in df.columns]

    temp_df = df[used_cols]
    imputed_array = imputer.fit_transform(temp_df)

    df['dcoilwtico'] = imputed_array[:, used_cols.index('dcoilwtico')]
    return df

train = fill_oil_with_iterative_imputer(train)

# Final cleaned dataframe ready!
train.head()

In [ ]:
train.tail()

In [ ]:
print(f"Train data shape after merging {train.shape}")

In [ ]:
test_df = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/test.csv')

In [ ]:
print(f"Test data {test_df.shape}")

In [ ]:
train['date'] = pd.to_datetime(train['date'])

train['day'] = train['date'].dt.day
train['month'] = train['date'].dt.month
train['year'] = train['date'].dt.year
train['weekday'] = train['date'].dt.weekday
train['is_weekend'] = train['weekday'].isin([5, 6]).astype(int)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
for col in ['family', 'city', 'state', 'type']:
    train[col] = le.fit_transform(train[col])

In [ ]:
features = [
    'store_nbr', 'family', 'onpromotion', 'transactions',
    'dcoilwtico', 'holiday_flag', 'cluster', 'type',
    'day', 'month', 'year', 'weekday', 'is_weekend'
]

target = 'sales'

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error
import numpy as np

X = train[features]
y = train[target]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Sort by date
train = train.sort_values('date')

# Set a cutoff date for validation — e.g., last 3 months of data for validation
cutoff_date = '2017-05-01'

# Time-based split
X_train = train[train['date'] < cutoff_date][features]
y_train = train[train['date'] < cutoff_date][target]

X_val = train[train['date'] >= cutoff_date][features]
y_val = train[train['date'] >= cutoff_date][target]


In [ ]:
import matplotlib.pyplot as plt

train.groupby('date')['sales'].sum().plot(figsize=(12,4), title="Total Sales Over Time")
plt.axvline(pd.to_datetime(cutoff_date), color='red', linestyle='--', label='Validation Start')
plt.legend()
plt.show()


In [ ]:
# Step 1: Load the test set
test = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/test.csv')  # should contain ['id', 'date', 'store_nbr', 'family', 'onpromotion']

# Step 2: Merge with stores
test = test.merge(stores, on='store_nbr', how='left')

# Step 3: Merge with oil prices
test = test.merge(oil[['date', 'dcoilwtico']], on='date', how='left')

# Step 4: Merge with holidays
test = test.merge(holidays[['date', 'holiday_flag']], on='date', how='left')
test['holiday_flag'] = test['holiday_flag'].fillna(0)

# Step 5: Merge with transactions
test = test.merge(transactions, on=['date', 'store_nbr'], how='left')
test['transactions'] = test['transactions'].fillna(0)


In [ ]:
# Fill missing oil prices using IterativeImputer (same as training)
def fill_oil_with_iterative_imputer_test(df, train_ref):
    imputer = IterativeImputer(random_state=42)
    used_cols = ['dcoilwtico', 'transactions', 'onpromotion']

    # Use training data to fit imputer
    imputer.fit(train_ref[used_cols])

    imputed_array = imputer.transform(df[used_cols])
    df['dcoilwtico'] = imputed_array[:, used_cols.index('dcoilwtico')]
    return df

test = fill_oil_with_iterative_imputer_test(test, train)


In [ ]:
test['date'] = pd.to_datetime(test['date'])
test['day'] = test['date'].dt.day
test['month'] = test['date'].dt.month
test['year'] = test['date'].dt.year
test['weekday'] = test['date'].dt.weekday
test['is_weekend'] = test['weekday'].isin([5, 6]).astype(int)


In [ ]:
# Apply same label encoding (train should include all possible categories!)
for col in ['family', 'city', 'state', 'type']:
    test[col] = le.fit_transform(test[col])


# Models with optuna

In [ ]:
import pandas as pd
import numpy as np
import optuna
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import make_scorer, mean_squared_log_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# -------------------------------
# Sample time-sorted data
# -------------------------------
train_sorted = train.sort_values("date")
train_sampled = train_sorted.iloc[:500_000]  # Adjust to 3_000_000 if needed

X_sample = train_sampled[features]
y_sample = train_sampled[target]

# -------------------------------
# RMSLE scorer
# -------------------------------
def rmsle(y_true, y_pred):
    y_pred = np.maximum(0, y_pred)
    return np.sqrt(mean_squared_log_error(y_true, y_pred))

rmsle_scorer = make_scorer(rmsle, greater_is_better=False)

# -------------------------------
# Optuna objective
# -------------------------------
def objective(trial):
    model_name = trial.suggest_categorical("model", ["xgb"])

    model_name == "xgb"
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 5),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 5),
        'random_state': 42,
        'n_jobs': -1,
        'tree_method': 'hist',
        'device': 'cuda'
        }
    model = XGBRegressor(**params, verbosity=0)

  
    # Time series cross-validation
    cv = TimeSeriesSplit(n_splits=3)
    score = cross_val_score(model, X_sample, y_sample, cv=cv, scoring=rmsle_scorer).mean()
    return score  # Negative RMSLE

# -------------------------------
# Run Optuna
# -------------------------------
study = optuna.create_study(direction='maximize')  # Because score is negative RMSLE
study.optimize(objective, n_trials=50)

# -------------------------------
# Best model and parameters
# -------------------------------
print("Best Trial:")
print(study.best_trial)

trial_params = study.best_trial.params.copy()
best_model_name = trial_params.pop("model")
best_params = trial_params

In [ ]:
# -------------------------------
# Train best model on full training set
# -------------------------------
best_model_name == "xgb"
final_model = XGBRegressor(**best_params, verbosity=0, tree_method="hist", device="cuda")

final_model.fit(X_train, y_train)

In [ ]:
# -------------------------------
# Validation RMSLE
# -------------------------------
y_val_pred = final_model.predict(X_val)
val_rmsle = rmsle(y_val, y_val_pred)
print(f"\nFinal Model: {best_model_name.upper()} | Validation RMSLE: {val_rmsle:.4f}")

In [ ]:
X_test = test[features]

# Predict with trained model
test_preds = final_model.predict(X_test)
test_preds = np.maximum(0, test_preds)  # Ensure no negative values

# Prepare submission
submission = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/sample_submission.csv')
submission['sales'] = test_preds

submission.to_csv('submission.csv', index=False)